# Profile Eva encoder FLOPs

Measure one encoder forward with two PyTorch counters and compare Eva and VirTues at the same input size.
The benchmark uses one random float32 image per forward in evaluation mode with gradients disabled. With 8 × 8 tokens, each input channel contributes 256 spatial tokens at 128 × 128 and 784 at 224 × 224. Synthetic marker identities can repeat, so the high-channel sweep measures arithmetic cost.

In [ ]:
from pathlib import Path
import os
import sys

import torch
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
from torch.profiler import ProfilerActivity, profile
from torch.utils.flop_counter import FlopCounterMode

eva_root = Path.cwd()
if eva_root.name == "tutorials":
    eva_root = eva_root.parent
if not (eva_root / "Eva" / "eva.py").is_file():
    raise FileNotFoundError("Open this notebook from the Eva repository or its tutorials directory.")
os.chdir(eva_root)  # Eva loads the GenePT file relative to this directory.

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
RESOLUTIONS = (128, 224)
CHANNELS = (10, 40, 100)

EVA_CHECKPOINT = os.environ.get("EVA_CHECKPOINT")
VIRTUES_REPO = Path("/path/to/virtues")
VIRTUES_MARKERS = None
VIRTUES_CHECKPOINT = None

print(f"torch={torch.__version__}, CUDA={torch.version.cuda}, device={DEVICE}")

torch=2.9.1+cu128, CUDA=12.8, device=cuda:0


## Count FLOPs on the same forward path

We measured FLOPs with [`FlopCounterMode`](https://github.com/pytorch/pytorch/blob/v2.11.0/torch/utils/flop_counter.py) and [`torch.profiler(with_flops=True)`](https://docs.pytorch.org/docs/2.11/profiler.html#torch.profiler.profile) for comparison.

In [26]:
@torch.no_grad()
def measure_forward(forward, device):
    activities = [ProfilerActivity.CPU]
    if device.type == "cuda":
        activities.append(ProfilerActivity.CUDA)

    with FlopCounterMode(display=False) as counter:
        forward()

    with profile(activities=activities, record_shapes=True, with_flops=True) as profiler:
        forward()

    counter_ops = {str(op): count for op, count in counter.get_flop_counts()["Global"].items()}
    profiler_ops = {event.key: event.flops for event in profiler.key_averages() if event.flops}
    return {
        "flopcounter_gflops": counter.get_total_flops() / 1e9,
        "profiler_gflops": sum(profiler_ops.values()) / 1e9,
        "counter_ops": counter_ops,
        "profiler_ops": profiler_ops,
    }

## Load the encoders

Eva's configuration calls the full crop size `ds.patch_size` and the spatial token size `ds.token_size`. VirTues uses the same model at both sizes. Its `forward_list` accepts the raw channel-first image and marker IDs.

The comparison loads both models. Enter the path to your VirTues checkout in `VIRTUES_REPO` in the settings cell. Put its ESM marker embeddings in the default directory or set `VIRTUES_MARKERS`. The loader uses the local sp32 checkpoint when available and downloads it from Hugging Face otherwise. [VirTues setup](https://github.com/bunnelab/virtues#readme)

In [27]:
from Eva.eva import EvaMAE

def load_eva_checkpoint():
    path = EVA_CHECKPOINT or hf_hub_download("yandrewl/Eva", filename="Eva_model.ckpt")
    checkpoint = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    return checkpoint.get("state_dict", checkpoint)

def build_eva(resolution, state_dict):
    config = OmegaConf.load(eva_root / "config.yaml")
    config.ds.patch_size = resolution
    config.ds.mask_ratio = 0.0
    model = EvaMAE(config)
    if resolution == 224:
        model.load_state_dict(state_dict)
    else:
        position_keys = {"model.enc_pos_embed", "model.decoder_pos_embed"}
        weights = {name: value for name, value in state_dict.items() if name not in position_keys}
        missing, unexpected = model.load_state_dict(weights, strict=False)
        if set(missing) != position_keys or unexpected:
            raise RuntimeError(f"Checkpoint mismatch: missing={missing}, unexpected={unexpected}")
    return model.to(DEVICE).eval()

In [28]:
def build_virtues():
    from safetensors.torch import load_file

    repo = VIRTUES_REPO.expanduser().resolve()
    model_config = OmegaConf.load(repo / "configs" / "base_config.yaml").model
    sys.path.insert(0, str(repo))
    from modules.multiplex_virtues import MultiplexVirtues
    from utils.utils import load_marker_embeddings

    marker_dir = (
        Path(VIRTUES_MARKERS).expanduser()
        if VIRTUES_MARKERS else repo / "protein_embeddings" / "esm2_t30_150M_UR50D"
    )
    if not any(marker_dir.glob("*.pt")):
        raise FileNotFoundError(f"Set VIRTUES_MARKERS to the directory containing VirTues marker embeddings. Looked in {marker_dir}.")
    local_checkpoint = repo / "assets" / "checkpoints" / "model-sp32.safetensors"
    checkpoint = VIRTUES_CHECKPOINT
    if checkpoint is None:
        checkpoint = local_checkpoint if local_checkpoint.is_file() else hf_hub_download(
            "bunnelab/virtues", filename="model-sp32.safetensors"
        )
    model = MultiplexVirtues(
        use_default_config=False,
        custom_config=None,
        prior_bias_embeddings=load_marker_embeddings(str(marker_dir)),
        prior_bias_embedding_type="esm",
        prior_bias_embedding_fusion_type="add",
        patch_size=model_config.patch_size,
        model_dim=model_config.model_dim,
        feedforward_dim=model_config.feedforward_dim,
        encoder_pattern=model_config.encoder_pattern,
        num_encoder_heads=model_config.num_encoder_heads,
        decoder_pattern=model_config.decoder_pattern,
        num_decoder_heads=model_config.num_decoder_heads,
        num_hidden_layers=model_config.num_decoder_hidden_layers,
        positional_embedding_type=model_config.positional_embedding_type,
        dropout=model_config.dropout,
        group_layers=model_config.group_layers,
        norm_after_encoder_decoder=model_config.norm_after_encoder_decoder,
        verbose=False,
    )
    model.load_state_dict(load_file(str(checkpoint)))
    return model.to(DEVICE).eval()

In [29]:
import pandas as pd

torch.manual_seed(42)
rows = []

def record(model_name, resolution, channels, forward):
    rows.append({
        "model": model_name,
        "resolution": resolution,
        "channels": channels,
        **measure_forward(forward, DEVICE),
    })

eva_weights = load_eva_checkpoint()
for resolution in RESOLUTIONS:
    eva = build_eva(resolution, eva_weights)
    for channels in CHANNELS:
        image = torch.randn(1, channels, resolution, resolution, device=DEVICE)
        markers = [["CD4"] * channels]
        record("Eva", resolution, channels, lambda: eva.model.forward_encoder(image, markers))
    del eva
del eva_weights

virtues = build_virtues()
vocabulary_size = virtues.encoder.prior_bias_embeddings.shape[0]
for resolution in RESOLUTIONS:
    for channels in CHANNELS:
        image = torch.randn(channels, resolution, resolution, device=DEVICE)
        marker_ids = torch.arange(channels, device=DEVICE) % vocabulary_size
        record(
            "VirTues", resolution, channels,
            lambda: virtues.encoder.forward_list([image], [marker_ids]),
        )
del virtues

results = pd.DataFrame(rows)
count_columns = ["flopcounter_gflops", "profiler_gflops"]
results.pivot(index=["resolution", "channels"], columns="model", values=count_columns).round(2)

flopcounter_gflops          profiler_gflops         
model                              Eva  VirTues             Eva  VirTues
resolution channels                                                     
128        10                    93.42   201.47           91.00   189.27
           40                   224.97   756.15          222.59   705.51
           100                  493.74  1888.14          491.43  1737.98
224        10                   301.00   691.60          278.34   579.62
           40                   703.88  2593.71          681.33  2160.56
           100                 1526.99  6467.31         1504.66  5322.44

In [30]:
def average_gflops_increase_per_channel(results, low=10, high=40):
    selected = results[results["channels"].isin((low, high))]
    table = selected.pivot(index=["model", "resolution"], columns="channels", values=count_columns)
    return ((table.xs(high, axis=1, level="channels") - table.xs(low, axis=1, level="channels"))
            / (high - low))

average_gflops_increase_per_channel(results).round(2)

flopcounter_gflops  profiler_gflops
model   resolution                                     
Eva     128                       4.39             4.39
        224                      13.43            13.43
VirTues 128                      18.49            17.21
        224                      63.40            52.70